<a href="https://colab.research.google.com/github/melinahou/Agent_1/blob/main/pipeline/pipelineAgent1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ----------------------------------------------------------------------------
# 0. Imports & global config
# ----------------------------------------------------------------------------
import os, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from pathlib import Path
from sklearn import set_config
from sklearn.impute import KNNImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold, GridSearchCV,
    cross_validate, cross_val_predict,
    train_test_split
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    brier_score_loss, average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve,
    precision_score, recall_score, f1_score,
    confusion_matrix
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

set_config(transform_output="pandas")

RANDOM_STATE = 42

RAW_SENSORS = [
    'Air_temperature_K',
    'Process_temperature_K',
    'Rotational_speed_rpm',
    'Torque_Nm',
    'Tool_wear_min'
]

TYPE_MAP = {
    'L': 0,
    'M': 1,
    'H': 2
}


# ----------------------------------------------------------------------------
# Global publication-grade figure style
# ----------------------------------------------------------------------------
FIG_DIR = Path('/content/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

FIG_SIZE = (6, 4)
FIG_SIZE_WIDE = (12, 5)
FIG_SIZE_TALL = (6, 8)

CM_CMAP = 'Blues'
MISSING_CMAP = 'Blues'
CORR_CMAP = 'vlag'

sns.set_theme(
    context='notebook',
    style='whitegrid',
    palette='colorblind'
)

plt.rcParams.update({
    'figure.figsize': FIG_SIZE,
    'font.size': 10,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})


def save_figure(fig, filename):
    """Save figure as PNG 300dpi + PDF + SVG."""

    fig.tight_layout()

    fig.savefig(
        FIG_DIR / f'{filename}.png',
        dpi=300
    )

    fig.savefig(
        FIG_DIR / f'{filename}.pdf'
    )

    fig.savefig(
        FIG_DIR / f'{filename}.svg'
    )

In [3]:
# ============================================================================
# 1. Dataset Loading & Structural Preprocessing (No Imputation Here)
# ============================================================================
def load_pmdi(path):
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y %H:%M')
    df = df.sort_values(['System', 'Date']).reset_index(drop=True)
    df.columns = (df.columns.str.replace('(', '', regex=False)
                            .str.replace(')', '', regex=False)
                            .str.replace(' ', '_', regex=False))
    df['Type'] = df['Type'].map(TYPE_MAP)
    return df

# ----------------------------------------------------------------------------
# Execute loading and basic inspection
# ----------------------------------------------------------------------------
df = load_pmdi('/content/AI4I-PMDI.csv')

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (10000, 12)

Columns:
['UDI', 'Date', 'System', 'Control', 'Product_ID', 'Type', 'Air_temperature_K', 'Process_temperature_K', 'Rotational_speed_rpm', 'Torque_Nm', 'Tool_wear_min', 'Diagnostic']

First 5 rows:


,UDI,Date,System,Control,Product_ID,Type,Air_temperature_K,Process_temperature_K,Rotational_speed_rpm,Torque_Nm,Tool_wear_min,Diagnostic
0,2,2014-04-12 16:09:00,0,A,L47181,0,298.2,308.7,1408.0,NaN,NaN,No failure
1,3,2014-04-13 01:13:00,0,A,L47182,0,298.1,308.5,1498.0,NaN,NaN,No failure
2,1,2014-04-15 11:56:00,0,C,M14860,1,NaN,NaN,NaN,42.8,0.0,No failure
3,8,2014-07-22 01:24:00,0,B,L47187,0,NaN,NaN,1527.0,40.2,NaN,No failure
4,5,2014-07-22 01:31:00,0,C,L47184,0,NaN,NaN,NaN,40.0,9.0,No failure


In [4]:
# ----------------------------------------------------------------------------
# 1.1 Exploratory Data Analysis: Diagnostics, controls, and missingness patterns
# ----------------------------------------------------------------------------

print("Diagnostic distribution:")
print(df['Diagnostic'].value_counts(dropna=False))

print("\nControl distribution:")
print(df['Control'].value_counts(dropna=False))

print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False))

sensor_cols = [
    'Air_temperature_K',
    'Process_temperature_K',
    'Rotational_speed_rpm',
    'Torque_Nm',
    'Tool_wear_min'
]

missing_by_control = (
    df.groupby('Control')[sensor_cols]
      .apply(lambda x: x.isna().mean() * 100)
)

print("Missing values (%) by Control:")
display(missing_by_control.round(2))

Diagnostic distribution:
Diagnostic
No failure                  9652
Heat Dissipation Failure     106
Overstrain Failure            98
Power Failure                 83
Tool Wear Failure             42
Random Failures               19
Name: count, dtype: int64

Control distribution:
Control
A    3437
C    3321
B    3242
Name: count, dtype: int64

Missing values per column:
Tool_wear_min            6679
Process_temperature_K    6563
Air_temperature_K        6563
Torque_Nm                3437
Rotational_speed_rpm     3321
System                      0
Date                        0
UDI                         0
Control                     0
Type                        0
Product_ID                  0
Diagnostic                  0
dtype: int64
Missing values (%) by Control:


,Air_temperature_K,Process_temperature_K,Rotational_speed_rpm,Torque_Nm,Tool_wear_min
Control,,,,,
A,0.0,0.0,0.0,100.0,100.0
B,100.0,100.0,0.0,0.0,100.0
C,100.0,100.0,100.0,0.0,0.0


In [5]:
# ----------------------------------------------------------------------------
# 2. Feature Engineering
# ----------------------------------------------------------------------------
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Δημιουργεί νέα χαρακτηριστικά (όπως Temp_diff_K και Torque_x_Wear)
    διατηρώντας παράλληλα τις υπάρχουσες στήλες και τους δείκτες απουσίας.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()

        # Υπολογισμός διαφοράς θερμοκρασίας (αν υπάρχουν οι αντίστοιχες στήλες)
        if 'Process_temperature_K' in X_out.columns and 'Air_temperature_K' in X_out.columns:
            X_out['Temp_diff_K'] = X_out['Process_temperature_K'] - X_out['Air_temperature_K']

        # Υπολογισμός ροπής επί φθοράς (Torque x Wear)
        if 'Torque_[Nm]' in X_out.columns and 'Tool_wear_[min]' in X_out.columns:
            X_out['Torque_x_Wear'] = X_out['Torque_[Nm]'] * X_out['Tool_wear_[min]']

        return X_out

# Ορισμός του αντικειμένου που θα χρησιμοποιηθεί στο pipeline
FEATURE_ENGINEER = FeatureEngineer()



In [6]:
# -------------------------------------------------------------------
# 2.1 Leakage-Safe Pipeline Construction
# -------------------------------------------------------------------

class MissingIndicatorAdder(BaseEstimator, TransformerMixin):
    """
    Προσθέτει binary indicators για τις στήλες θερμοκρασίας
    πριν ο KNNImputer συμπληρώσει τις κενές τιμές (Control B & C).
    """
    def __init__(self, cols_to_check=None):
        self.cols_to_check = cols_to_check

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        if self.cols_to_check:
            for col in self.cols_to_check:
                if col in X_out.columns:
                    # Δημιουργία δείκτη: 1 αν λείπει η τιμή, 0 αλλιώς
                    X_out[f'{col}_isnan'] = X_out[col].isna().astype(int)
        return X_out

# Ορισμός των στηλών θερμοκρασίας που λείπουν στα Controls B και C
temperature_cols = ['Air_temperature_K', 'Process_temperature_K']

def build_pipeline(clf, use_smote=False) -> ImbPipeline:
    """
    Leakage-safe pipeline διορθωμένο ώστε να προσθέτει missing indicators
    πριν το KNN imputation.
    """
    steps = [
        # 1. Προσθήκη δεικτών απουσίας για τις θερμοκρασίες
        ('missing_indicators', MissingIndicatorAdder(cols_to_check=temperature_cols)),
        # 2. KNN Imputation ασφαλές μέσα στο fold
        ('imputer', KNNImputer(n_neighbors=5)),
        # 3. Feature Engineering (περιλαμβάνει το Temp_diff_K και Torque_x_Wear)
        ('features', FEATURE_ENGINEER),
    ]

    if use_smote:
        steps.append(
            ('smote', SMOTE(random_state=RANDOM_STATE))
        )

    steps.append(
        ('clf', clf)
    )

    return ImbPipeline(steps=steps)



In [7]:
# ============================================================================
# 3. Target Definition (Excluding Random Failures for Agent 1 Training)
# ============================================================================
def define_target(df):
    """
    Binary target για τον Agent 1.

    0 = No Failure
    1 = Predictable Failure

    Τα Random Failures εξαιρούνται από το supervised training.
    """

    # Κρατάμε όλες τις γραμμές εκτός από Random Failures
    model_mask = df['Diagnostic'] != 'Random Failures'

    df_model = df.loc[model_mask].copy()

    # Input features
    X = df_model[['Type'] + RAW_SENSORS].copy()

    # Binary target
    y = (df_model['Diagnostic'] != 'No failure').astype(int)

    return X, y

# ----------------------------------------------------------------------------
# Execute target definition and verify sizes/distributions
# ----------------------------------------------------------------------------
X, y = define_target(df)

print("Dataset size after RNF exclusion:", len(y))
print("\nBinary target distribution:")
print(y.value_counts())

print("\nBinary target percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

Dataset size after RNF exclusion: 9981

Binary target distribution:
Diagnostic
0    9652
1     329
Name: count, dtype: int64

Binary target percentages:
Diagnostic
0    96.7
1     3.3
Name: proportion, dtype: float64


In [8]:
# ============================================================================
# 4. Class Imbalance Strategy Comparison (GroupKFold Evaluation - Leakage Safe)
# ============================================================================

from sklearn.model_selection import GroupKFold, cross_validate


def compare_imbalance_strategies(X, y, groups):
    """
    Συγκρίνει στρατηγικές class imbalance χρησιμοποιώντας GroupKFold
    ανά System/UID (με n_splits=3 αφού έχουμε 3 ομάδες),
    αποτρέποντας το data leakage.
    """

    # Χρήση n_splits=3 επειδή οι διαθέσιμες ομάδες (groups) είναι συνολικά 3
    gkf = GroupKFold(n_splits=3)

    # Αρχικός λόγος negatives / positives
    spw = (y == 0).sum() / (y == 1).sum()

    print(f"scale_pos_weight = {spw:.2f}")

    strategies = {

        # 1. Χωρίς καμία ειδική διόρθωση (Plain XGBoost)
        'XGBoost plain': (
            XGBClassifier(
                scale_pos_weight=1.0,
                eval_metric='logloss',
                random_state=RANDOM_STATE
            ),
            False
        ),

        # 2. Μόνο class weighting
        'XGBoost weighted': (
            XGBClassifier(
                scale_pos_weight=spw,
                eval_metric='logloss',
                random_state=RANDOM_STATE
            ),
            False
        ),

        # 3. Μόνο SMOTE
        'XGBoost + SMOTE': (
            XGBClassifier(
                scale_pos_weight=1.0,
                eval_metric='logloss',
                random_state=RANDOM_STATE
            ),
            True
        ),

        # 4. SMOTE + weighting
        'XGBoost + SMOTE + weight': (
            XGBClassifier(
                scale_pos_weight=spw,
                eval_metric='logloss',
                random_state=RANDOM_STATE
            ),
            True
        )
    }

    scoring = [
        'precision',
        'recall',
        'f1',
        'roc_auc',
        'average_precision'
    ]

    results = []

    for name, (clf, use_smote) in strategies.items():

        print(f"\nRunning: {name}")

        scores = cross_validate(
            build_pipeline(
                clf,
                use_smote=use_smote
            ),
            X,
            y,
            cv=gkf,
            groups=groups,  # Ομαδοποίηση ανά σύστημα/μηχανή
            scoring=scoring,
            n_jobs=-1
        )

        results.append({
            'Strategy': name,
            'Precision': scores['test_precision'].mean(),
            'Recall': scores['test_recall'].mean(),
            'F1': scores['test_f1'].mean(),
            'ROC-AUC': scores['test_roc_auc'].mean(),
            'Average Precision': scores['test_average_precision'].mean()
        })

    results_df = pd.DataFrame(results)

    return results_df

# ----------------------------------------------------------------------------
# Execute imbalance strategy comparison with GroupKFold (3 splits)
# ----------------------------------------------------------------------------
groups_col = X['UID'] if 'UID' in X.columns else X.iloc[:, 0]

imbalance_results = compare_imbalance_strategies(X, y, groups=groups_col)

display(
    imbalance_results.round(4)
)

scale_pos_weight = 29.34

Running: XGBoost plain

Running: XGBoost weighted

Running: XGBoost + SMOTE

Running: XGBoost + SMOTE + weight


,Strategy,Precision,Recall,F1,ROC-AUC,Average Precision
0,XGBoost plain,0.7701,0.6654,0.7080,0.9892,0.8042
1,XGBoost weighted,0.6974,0.7618,0.7225,0.9895,0.8303
2,XGBoost + SMOTE,0.6276,0.8356,0.6895,0.9911,0.7933
3,XGBoost + SMOTE + weight,0.5615,0.8501,0.6528,0.9897,0.7860


In [9]:
# ----------------------------------------------------------------------------
# 5. Baseline Model Comparison (LogReg vs Random Forest vs XGBoost)
# ----------------------------------------------------------------------------

def compare_models(X, y, groups, spw):

    # Χρήση GroupKFold με n_splits=3 (αφού οι ομάδες μας είναι 3)
    gkf = GroupKFold(n_splits=3)

    models = {
        'Logistic Regression': LogisticRegression(
            max_iter=1000,
            class_weight='balanced'
        ),

        'Random Forest': RandomForestClassifier(
            n_estimators=300,
            class_weight='balanced',
            random_state=RANDOM_STATE
        ),

        'XGBoost': XGBClassifier(
            scale_pos_weight=spw,
            eval_metric='logloss',
            random_state=RANDOM_STATE
        )
    }

    scoring = [
        'precision',
        'recall',
        'f1',
        'roc_auc',
        'average_precision'
    ]

    results = []

    for name, model in models.items():

        print(f"Running: {name}")

        scores = cross_validate(
            build_pipeline(model, use_smote=False),
            X,
            y,
            cv=gkf,
            groups=groups,  # Ομαδοποίηση ανά σύστημα/μηχανή για αποφυγή leakage
            scoring=scoring,
            n_jobs=-1
        )

        results.append({
            'Model': name,
            'Precision': scores['test_precision'].mean(),
            'Recall': scores['test_recall'].mean(),
            'F1': scores['test_f1'].mean(),
            'ROC-AUC': scores['test_roc_auc'].mean(),
            'Average Precision': scores['test_average_precision'].mean()
        })

    return pd.DataFrame(results)


# ----------------------------------------------------------------------------
# Run baseline model comparison with GroupKFold
# ----------------------------------------------------------------------------

spw = (y == 0).sum() / (y == 1).sum()

print(f"scale_pos_weight = {spw:.4f}")

# Ορισμός των ομάδων (systems/UIDs)
groups_col = X['UID'] if 'UID' in X.columns else X.iloc[:, 0]

model_results = compare_models(X, y, groups=groups_col, spw=spw)

display(model_results.round(4))

scale_pos_weight = 29.3374
Running: Logistic Regression
Running: Random Forest
Running: XGBoost


,Model,Precision,Recall,F1,ROC-AUC,Average Precision
0,Logistic Regression,0.0792,0.5760,0.1383,0.7918,0.1297
1,Random Forest,0.8895,0.6357,0.7379,0.9910,0.8376
2,XGBoost,0.6974,0.7618,0.7225,0.9895,0.8303


In [10]:
# ----------------------------------------------------------------------------
# 6. Rigorous Nested Cross-Validation & Hyperparameter Optimization
# ----------------------------------------------------------------------------
PARAM_GRID = {
    'clf__max_depth': [3, 5],
    'clf__n_estimators': [100, 200],
    'clf__learning_rate': [0.05, 0.1],
}
SCORING = ['precision', 'recall', 'f1', 'roc_auc', 'average_precision']


def nested_cv_evaluate(X, y, groups, spw):
    # Εξωτερικός βρόχος: GroupKFold ανά σύστημα (αποφυγή leakage στο test)
    outer = GroupKFold(n_splits=3)

    # Εσωτερικός βρόχος: StratifiedKFold για να μην πέφτουμε σε folds χωρίς βλάβες στο tuning
    inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    base = XGBClassifier(
        scale_pos_weight=spw,
        eval_metric='logloss',
        random_state=RANDOM_STATE
    )

    search = GridSearchCV(
        build_pipeline(base, use_smote=False),
        PARAM_GRID,
        scoring='average_precision',
        cv=inner,
        n_jobs=-1
    )

    res = cross_validate(
        search,
        X,
        y,
        cv=outer,
        groups=groups,  # Ομαδοποίηση ανά σύστημα στον εξωτερικό βρόχο
        scoring=SCORING,
        n_jobs=-1
    )

    print("=== Nested-CV (Leakage-Safe ανά System) ===")

    for m in SCORING:
        s = res[f'test_{m}']
        print(
            f"  {m:<18}: "
            f"{s.mean():.4f} ± {s.std():.4f}"
        )

    return res


def collect_fold_curves(pipe, X, y, groups):
    """Per-fold ROC & PR curves με GroupKFold — για SOTA καμπύλες με σκιασμένη ζώνη."""
    cv = GroupKFold(n_splits=3)
    fold_data = []

    for tr, te in cv.split(X, y, groups=groups):
        pipe.fit(X.iloc[tr], y.iloc[tr])
        p = pipe.predict_proba(X.iloc[te])[:, 1]
        fpr, tpr, _ = roc_curve(y.iloc[te], p)
        prec, rec, _ = precision_recall_curve(y.iloc[te], p)
        fold_data.append({
            'fpr': fpr, 'tpr': tpr, 'prec': prec, 'rec': rec,
            'auc': roc_auc_score(y.iloc[te], p),
            'ap': average_precision_score(y.iloc[te], p)
        })
    return fold_data   # -> figures: ROC & PR με mean ± std band

# ----------------------------------------------------------------------------
# Εκτέλεση Nested CV
# ----------------------------------------------------------------------------
groups_col = X['UID'] if 'UID' in X.columns else X.iloc[:, 0]

nested_results = nested_cv_evaluate(X, y, groups=groups_col, spw=spw)

=== Nested-CV (Leakage-Safe ανά System) ===
  precision         : 0.5917 ± 0.0428
  recall            : 0.8295 ± 0.0465
  f1                : 0.6879 ± 0.0114
  roc_auc           : 0.9846 ± 0.0131
  average_precision : 0.8358 ± 0.0123


In [11]:
# ----------------------------------------------------------------------------
# 6.1 Execute Nested Cross-Validation evaluation
# ----------------------------------------------------------------------------

spw = (y == 0).sum() / (y == 1).sum()

# Ορισμός των ομάδων (systems/UIDs) για τον GroupKFold
groups_col = X['UID'] if 'UID' in X.columns else X.iloc[:, 0]

nested_res = nested_cv_evaluate(
    X=X,
    y=y,
    groups=groups_col,  # Προσθήκη των groups εδώ
    spw=spw
)

=== Nested-CV (Leakage-Safe ανά System) ===
  precision         : 0.5917 ± 0.0428
  recall            : 0.8295 ± 0.0465
  f1                : 0.6879 ± 0.0114
  roc_auc           : 0.9846 ± 0.0131
  average_precision : 0.8358 ± 0.0123


In [12]:
# ----------------------------------------------------------------------------
# 7. Final Hyperparameter Optimization on Full Dataset & Curve Collection
# ----------------------------------------------------------------------------

# 1. Εκτέλεση Grid Search στο σύνολο των δεδομένων (με GroupKFold ανά σύστημα)
spw = (y == 0).sum() / (y == 1).sum()

# Ορισμός των ομάδων (systems/UIDs)
groups_col = X['UID'] if 'UID' in X.columns else X.iloc[:, 0]

final_search = GridSearchCV(
    estimator=build_pipeline(
        XGBClassifier(
            scale_pos_weight=spw,
            eval_metric='logloss',
            random_state=RANDOM_STATE
        ),
        use_smote=False
    ),
    param_grid=PARAM_GRID,
    scoring='average_precision',
    cv=GroupKFold(n_splits=3),  # Διορθώθηκε σε GroupKFold(n_splits=3)
    n_jobs=-1
)

final_search.fit(X, y, groups=groups_col)  # Πέρασμα των groups στο fit

# 2. Αυτόματη εξαγωγή βέλτιστων παραμέτρων (χωρίς hardcoding)
best = {
    k.replace('clf__', ''): v
    for k, v in final_search.best_params_.items()
}

print("Best hyperparameters found:", best)
print(f"Best CV Average Precision: {final_search.best_score_:.4f}\n")

# 3. Κατασκευή τελικού pipeline και συλλογή fold curves για τα SOTA plots
best_model = XGBClassifier(
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    **best
)

best_pipe = build_pipeline(
    best_model,
    use_smote=False
)

# Πέρασμα των groups και στη συνάρτηση συλλογής καμπυλών
fold_data = collect_fold_curves(
    best_pipe,
    X,
    y,
    groups=groups_col
)

print(f"Successfully collected data from {len(fold_data)} folds.")
for i, fold in enumerate(fold_data, start=1):
    print(f"  Fold {i}: ROC-AUC={fold['auc']:.4f}, AP={fold['ap']:.4f}")


KeyboardInterrupt: 

In [ ]:
# ----------------------------------------------------------------------------
# 8. Probability Calibration (Uncalibrated vs Sigmoid vs Isotonic)
# ----------------------------------------------------------------------------
def compute_ece(y_true, y_prob, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        m = (y_prob >= edges[i]) & (y_prob < edges[i + 1])
        if m.sum():
            ece += (m.sum() / len(y_prob)) * abs(y_prob[m].mean() - y_true[m].mean())
    return ece


def oof_probs(X, y, groups, best, spw):
    """
    Συγκρίνει:
    1. Uncalibrated probabilities
    2. Sigmoid calibration
    3. Isotonic calibration
    Όλα υπολογίζονται με out-of-fold predictions με βάση τα συστήματα (GroupKFold).
    """

    # Χρήση GroupKFold ανά σύστημα (n_splits=3)
    cv = GroupKFold(n_splits=3)

    clf = XGBClassifier(
        scale_pos_weight=spw,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        **best
    )

    # 1. Uncalibrated probabilities
    # ---------------------------------------------------------
    p_uncal = cross_val_predict(
        build_pipeline(clf, use_smote=False),
        X,
        y,
        cv=cv,
        groups=groups,
        method='predict_proba',
        n_jobs=-1
    )[:, 1]

    # 2. Sigmoid calibration
    # ---------------------------------------------------------
    sigmoid_cal = CalibratedClassifierCV(
        estimator=build_pipeline(clf, use_smote=False),
        method='sigmoid',
        cv=3
    )

    p_sigmoid = cross_val_predict(
        sigmoid_cal,
        X,
        y,
        cv=cv,
        groups=groups,
        method='predict_proba',
        n_jobs=-1
    )[:, 1]

    # 3. Isotonic calibration
    # ---------------------------------------------------------
    isotonic_cal = CalibratedClassifierCV(
        estimator=build_pipeline(clf, use_smote=False),
        method='isotonic',
        cv=3
    )

    p_isotonic = cross_val_predict(
        isotonic_cal,
        X,
        y,
        cv=cv,
        groups=groups,
        method='predict_proba',
        n_jobs=-1
    )[:, 1]

    # ---------------------------------------------------------
    # Metrics
    # ---------------------------------------------------------
    results = pd.DataFrame({
        'Method': [
            'Uncalibrated',
            'Sigmoid',
            'Isotonic'
        ],
        'Brier': [
            brier_score_loss(y, p_uncal),
            brier_score_loss(y, p_sigmoid),
            brier_score_loss(y, p_isotonic)
        ],
        'ECE': [
            compute_ece(y.values, p_uncal),
            compute_ece(y.values, p_sigmoid),
            compute_ece(y.values, p_isotonic)
        ]
    })

    print("=== Calibration comparison ===")
    display(results.round(4))

    return p_uncal, p_sigmoid, p_isotonic, results

In [ ]:
# ----------------------------------------------------------------------------
# 8.1 Execute calibration comparison and select Isotonic method
# ----------------------------------------------------------------------------

# Ορισμός των ομάδων (systems/UIDs) για τον GroupKFold
groups_col = X['UID'] if 'UID' in X.columns else X.iloc[:, 0]

p_uncal, p_sigmoid, p_isotonic, calibration_results = oof_probs(
    X=X,
    y=y,
    groups=groups_col,  # Προσθήκη των groups εδώ
    best=best,
    spw=spw
)

p_cal = p_isotonic

CALIBRATION_METHOD = 'isotonic'

print(f"Selected calibration method: {CALIBRATION_METHOD}")

In [ ]:
# ----------------------------------------------------------------------------
# 9. Cost-Sensitive Decision Analysis & Optimal Threshold Selection
# ----------------------------------------------------------------------------
def cost_curve(y_true, y_prob, c_fn=10.0, c_fp=1.0):
    ts = np.linspace(0.01, 0.99, 99)
    costs = np.array([
        c_fn * ((y_true == 1) & (y_prob < t)).sum() +
        c_fp * ((y_true == 0) & (y_prob >= t)).sum() for t in ts])
    t_star = float(ts[int(costs.argmin())])
    print(f"Cost-optimal threshold (C_FN={c_fn}, C_FP={c_fp}): {t_star:.2f}")
    return ts, costs, t_star   # -> figure: expected cost vs threshold, mark t*


def cost_sensitivity(y_true, y_prob, ratios=(2, 5, 10, 20, 50)):
    """t* ως συνάρτηση του λόγου C_FN/C_FP — για SOTA sensitivity figure."""
    out = {}
    for r in ratios:
        _, _, t = cost_curve(y_true, y_prob, c_fn=r, c_fp=1.0)
        out[r] = t
    return out   # -> figure: t* vs cost-ratio


# Execute cost analysis and find optimal decision threshold
# ----------------------------------------------------------------------------
C_FN = 10.0
C_FP = 1.0

ts, costs, t_star = cost_curve(
    y_true=y.values,
    y_prob=p_cal,
    c_fn=C_FN,
    c_fp=C_FP
)

print(f"\nSelected threshold: {t_star:.2f}")

In [ ]:
# ----------------------------------------------------------------------------
# 9.1 Evaluate performance at selected threshold
# ----------------------------------------------------------------------------

# 1. Αξιολόγηση απόδοσης στο βέλτιστο threshold (t_star)
y_pred_cost = (p_cal >= t_star).astype(int)

precision_t = precision_score(y, y_pred_cost)
recall_t = recall_score(y, y_pred_cost)
f1_t = f1_score(y, y_pred_cost)

fp = ((y == 0) & (y_pred_cost == 1)).sum()
fn = ((y == 1) & (y_pred_cost == 0)).sum()
tp = ((y == 1) & (y_pred_cost == 1)).sum()
tn = ((y == 0) & (y_pred_cost == 0)).sum()

print(f"Threshold: {t_star:.2f}")
print(f"Precision: {precision_t:.4f}")
print(f"Recall:    {recall_t:.4f}")
print(f"F1:        {f1_t:.4f}")

print("\nConfusion counts:")
print(f"TP = {tp}")
print(f"FP = {fp}")
print(f"TN = {tn}")
print(f"FN = {fn}")

# 2. Ανάλυση ευαισθησίας κόστους (Cost Sensitivity Analysis)
cost_ratios = [2, 5, 10, 20, 50]

sens = cost_sensitivity(
    y_true=y.values,
    y_prob=p_cal,
    ratios=cost_ratios
)

print("\nCost sensitivity:")
for ratio, threshold in sens.items():
    print(
        f"C_FN / C_FP = {ratio:>2}  "
        f"-> optimal threshold = {threshold:.2f}"
    )


In [ ]:
# ----------------------------------------------------------------------------
# 9.2 Detailed Cost-Sensitivity Metrics Table Generation
# ----------------------------------------------------------------------------

sensitivity_rows = []

for ratio in cost_ratios:

    _, _, threshold = cost_curve(
        y_true=y.values,
        y_prob=p_cal,
        c_fn=ratio,
        c_fp=1.0
    )

    y_pred = (p_cal >= threshold).astype(int)

    tp = ((y == 1) & (y_pred == 1)).sum()
    fp = ((y == 0) & (y_pred == 1)).sum()
    tn = ((y == 0) & (y_pred == 0)).sum()
    fn = ((y == 1) & (y_pred == 0)).sum()

    sensitivity_rows.append({
        'C_FN/C_FP': ratio,
        'Threshold': threshold,
        'Precision': precision_score(y, y_pred),
        'Recall': recall_score(y, y_pred),
        'F1': f1_score(y, y_pred),
        'FP': fp,
        'FN': fn
    })

cost_sensitivity_df = pd.DataFrame(sensitivity_rows)

display(cost_sensitivity_df.round(4))

In [ ]:
# ----------------------------------------------------------------------------
# 9.3 Bootstrap Confidence Intervals for Threshold & Metrics
# ----------------------------------------------------------------------------
def bootstrap_evaluation(y_true, y_prob, c_fn=10.0, c_fp=1.0, n_bootstraps=500, random_state=42):
    rng = np.random.default_rng(random_state)
    n = len(y_true)

    boot_thresholds = []
    boot_precisions = []
    boot_recalls = []
    boot_f1s = []

    for _ in range(n_bootstraps):
        # Τυχαία δειγματοληψία με επιστροφή (Bootstrap resampling)
        indices = rng.choice(n, size=n, replace=True)
        y_b = y_true.iloc[indices] if hasattr(y_true, 'iloc') else y_true[indices]
        p_b = y_prob[indices]

        # Αν έχουμε μόνο μία κλάση στο δείγμα, παραλείπουμε την επανάληψη
        if len(np.unique(y_b)) < 2:
            continue

        # Υπολογισμός βέλτιστου threshold για το τρέχον bootstrap δείγμα
        ts = np.linspace(0.01, 0.99, 99)
        costs = np.array([
            c_fn * ((y_b == 1) & (p_b < t)).sum() +
            c_fp * ((y_b == 0) & (p_b >= t)).sum() for t in ts
        ])
        t_star_b = float(ts[int(costs.argmin())])

        # Προβλέψεις με βάση το νέο threshold
        y_pred_b = (p_b >= t_star_b).astype(int)

        # Υπολογισμός μετρήσεων
        prec = precision_score(y_b, y_pred_b, zero_division=0)
        rec = recall_score(y_b, y_pred_b, zero_division=0)
        f1 = f1_score(y_b, y_pred_b, zero_division=0)

        boot_thresholds.append(t_star_b)
        boot_precisions.append(prec)
        boot_recalls.append(rec)
        boot_f1s.append(f1)

    # Υπολογισμός διαστημάτων εμπιστοσύνης 95% (2.5% και 97.5%)
    results = {
        'Metric': ['Threshold (t*)', 'Precision', 'Recall', 'F1-Score'],
        'Mean': [
            np.mean(boot_thresholds),
            np.mean(boot_precisions),
            np.mean(boot_recalls),
            np.mean(boot_f1s)
        ],
        'CI 95% Lower': [
            np.percentile(boot_thresholds, 2.5),
            np.percentile(boot_precisions, 2.5),
            np.percentile(boot_recalls, 2.5),
            np.percentile(boot_f1s, 2.5)
        ],
        'CI 95% Upper': [
            np.percentile(boot_thresholds, 97.5),
            np.percentile(boot_precisions, 97.5),
            np.percentile(boot_recalls, 97.5),
            np.percentile(boot_f1s, 97.5)
        ]
    }

    return pd.DataFrame(results)

# Εκτέλεση για C_FN=10 και C_FP=1
y_arr = y.values if hasattr(y, 'values') else y

bootstrap_df = bootstrap_evaluation(
    y_true=y_arr,
    y_prob=p_cal,  # <-- Διορθώθηκε σε y_prob
    c_fn=10.0,
    c_fp=1.0,
    n_bootstraps=500
)

print("=== Bootstrap Confidence Intervals (95%) ===")
display(bootstrap_df.round(4))

In [ ]:
# ----------------------------------------------------------------------------
# 10 Mondrian Conformal Prediction Functions (Class-Conditional & Coverage Sweep)
# ----------------------------------------------------------------------------
def conformal_qhat_mondrian(y_true, p1, alpha=0.10):
    """
    Υπολογίζει ξεχωριστά conformal quantiles (qhat) ανά κλάση (Mondrian/Class-Conditional),
    προστατεύοντας τις πραγματικές βλάβες από το να αγνοούνται.
    """
    p = np.column_stack([1 - p1, p1])
    y_true = np.array(y_true)
    n = len(y_true)

    # Nonconformity scores για όλες τις παρατηρήσεις (1 - πιθανότητα αληθινής κλάσης)
    scores = 1 - p[np.arange(n), y_true]

    qhats = {}
    for cls in np.unique(y_true):
        cls_scores = scores[y_true == cls]
        n_cls = len(cls_scores)
        if n_cls > 0:
            q_level = np.ceil((n_cls + 1) * (1 - alpha)) / n_cls
            q_level = min(1.0, max(0.0, q_level))
            qhats[cls] = float(np.quantile(cls_scores, q_level, method='higher'))
        else:
            qhats[cls] = 1.0

    print(f"Mondrian Conformal qhats (alpha={alpha}): {qhats}")
    return qhats, scores

def conformal_sweep_mondrian(
    y_true,
    p1,
    alphas=np.linspace(0.01, 0.30, 15)
):
    """
    Ελέγχει την Mondrian conformal συμπεριφορά ανά κλάση για διαφορετικά alpha,
    καταγράφοντας την κάλυψη ξεχωριστά για τις βλάβες και τις υγιείς μηχανές.
    """
    p = np.column_stack([1 - p1, p1])
    y_true = np.array(y_true)
    n = len(y_true)

    coverage_class_0 = []
    coverage_class_1 = []
    overall_coverage = []
    mean_size = []
    empty_rate = []
    ambiguous_rate = []
    uncertainty_rate = []

    for alpha in alphas:
        # Υπολογισμός qhats ανά κλάση για το τρέχον alpha
        qhats, scores = conformal_qhat_mondrian(y_true, p1, alpha=alpha)

        # Prediction sets βασισμένα στα ξεχωριστά qhat ανά κλάση
        sets = np.zeros_like(p, dtype=bool)
        for cls, qhat_val in qhats.items():
            # Μια κλάση περιλαμβάνεται αν το score της είναι <= qhat της αντίστοιχης κλάσης
            class_scores = 1 - p[:, cls]
            sets[:, cls] = class_scores <= qhat_val

        set_sizes = sets.sum(axis=1)

        # Κάλυψη συνολικά και ανά κλάση
        covered = sets[np.arange(n), y_true].mean()
        overall_coverage.append(covered)

        # Κάλυψη για Κλάση 0 (υγιείς) και Κλάση 1 (βλάβες)
        if (y_true == 0).sum() > 0:
            cov_0 = sets[y_true == 0, 0].mean()
        else:
            cov_0 = 1.0
        coverage_class_0.append(cov_0)

        if (y_true == 1).sum() > 0:
            cov_1 = sets[y_true == 1, 1].mean()
        else:
            cov_1 = 1.0
        coverage_class_1.append(cov_1)

        # Average number of classes in the set
        mean_size.append(set_sizes.mean())

        # {} : καμία κλάση (Empty set)
        empty_rate.append((set_sizes == 0).mean())

        # {0,1} : και οι δύο κλάσεις (Ambiguous / escalation signal)
        ambiguous_rate.append((set_sizes == 2).mean())

        # Οτιδήποτε δεν είναι singleton (uncertainty)
        uncertainty_rate.append((set_sizes != 1).mean())

    return (
        alphas,
        np.array(overall_coverage),
        np.array(coverage_class_0),
        np.array(coverage_class_1),
        np.array(mean_size),
        np.array(empty_rate),
        np.array(ambiguous_rate),
        np.array(uncertainty_rate)
    )

In [ ]:
# ----------------------------------------------------------------------------
# 10.1 Execution of Mondrian Conformal Uncertainty & Coverage Sweep
# ----------------------------------------------------------------------------

# 1. Υπολογισμός βασικών qhats ανά κλάση για alpha = 0.10
qhats_dict, cscores = conformal_qhat_mondrian(
    y_true=y,
    p1=p_cal,
    alpha=0.10
)

print(f"Selected Mondrian Conformal qhats: {qhats_dict}")

# 2. Εκτέλεση sweep για πολλαπλά alphas με Mondrian
(
    alphas,
    overall_cov,
    cov_class_0,
    cov_class_1,
    size,
    empty,
    ambiguous,
    uncertain
) = conformal_sweep_mondrian(
    y_true=y,
    p1=p_cal
)

conformal_results = pd.DataFrame({
    'Alpha': alphas,
    'Nominal Coverage': 1.0 - alphas,
    'Overall Coverage': overall_cov,
    'Coverage (Class 0 - Healthy)': cov_class_0,
    'Coverage (Class 1 - Failure)': cov_class_1,
    'Mean Set Size': size,
    'Empty Set Rate': empty,
    'Ambiguous Set Rate': ambiguous,
    'Total Uncertainty Rate': uncertain
})

print("\n=== Mondrian Conformal Coverage & Uncertainty Sweep ===")
display(
    conformal_results.round(4)
)

In [ ]:
# ----------------------------------------------------------------------------
# 10.2 Proper Mondrian Split-Conformal Evaluation Protocol (Independent Test Set)
# ----------------------------------------------------------------------------

def split_conformal_evaluate(
    X,
    y,
    best,
    spw,
    alpha=0.10,
    test_size=0.20,
    calibration_size=0.20
):
    """
    Proper Mondrian (Class-Conditional) split-conformal protocol.

    1. Χωρίζει τα δεδομένα σε:
        - training
        - conformal calibration
        - independent test

    2. Εκπαιδεύει calibrated weighted XGBoost μόνο στο training set.

    3. Υπολογίζει ξεχωριστά q_hat ανά κλάση μόνο στο conformal calibration set.

    4. Ελέγχει κάλυψη (συνολική και ανά κλάση) και prediction-set behavior
       μόνο στο independent test set.
    """

    # 1. Split: development vs independent test
    # ---------------------------------------------------------
    X_dev, X_test, y_dev, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=RANDOM_STATE
    )

    # 2. Split development data: training vs conformal calibration
    # ---------------------------------------------------------
    calibration_fraction_of_dev = (
        calibration_size / (1.0 - test_size)
    )

    X_train, X_conf, y_train, y_conf = train_test_split(
        X_dev,
        y_dev,
        test_size=calibration_fraction_of_dev,
        stratify=y_dev,
        random_state=RANDOM_STATE
    )

    print("=== Split sizes ===")
    print(f"Training set:              {len(y_train)}")
    print(f"Conformal calibration set: {len(y_conf)}")
    print(f"Independent test set:      {len(y_test)}")

    # 3. Recalculate class weight ONLY from training data
    # ---------------------------------------------------------
    spw_train = (
        (y_train == 0).sum()
        / (y_train == 1).sum()
    )

    # 4. Tuned weighted XGBoost
    # ---------------------------------------------------------
    clf = XGBClassifier(
        scale_pos_weight=spw_train,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        **best
    )

    # 5. Isotonic calibrated model
    # ---------------------------------------------------------
    model = CalibratedClassifierCV(
        build_pipeline(
            clf,
            use_smote=False
        ),
        method='isotonic',
        cv=5
    )

    model.fit(
        X_train,
        y_train
    )

    # 6. Probabilities on conformal calibration set
    # ---------------------------------------------------------
    p_conf = model.predict_proba(
        X_conf
    )[:, 1]

    # 7. Compute Mondrian qhats ONLY on conformal calibration set
    # ---------------------------------------------------------
    p_conf_2class = np.column_stack([
        1 - p_conf,
        p_conf
    ])
    y_conf_arr = np.array(y_conf)
    conf_scores = (
        1 - p_conf_2class[
            np.arange(len(y_conf_arr)),
            y_conf_arr
        ]
    )

    qhats = {}
    for cls in np.unique(y_conf_arr):
        cls_scores = conf_scores[y_conf_arr == cls]
        n_cls = len(cls_scores)
        if n_cls > 0:
            q_level = min(np.ceil((n_cls + 1) * (1 - alpha)) / n_cls, 1.0)
            q_level = max(0.0, q_level)
            qhats[cls] = float(np.quantile(cls_scores, q_level, method='higher'))
        else:
            qhats[cls] = 1.0

    print(f"\nSplit-conformal Mondrian qhats (alpha={alpha}): {qhats}")

    # 8. Independent test probabilities
    # ---------------------------------------------------------
    p_test = model.predict_proba(
        X_test
    )[:, 1]

    p_test_2class = np.column_stack([
        1 - p_test,
        p_test
    ])
    y_test_arr = np.array(y_test)
    n_test = len(y_test_arr)

    # 9. Prediction sets on TEST using class-conditional qhats
    # ---------------------------------------------------------
    test_sets = np.zeros_like(p_test_2class, dtype=bool)
    for cls, qhat_val in qhats.items():
        class_scores = 1 - p_test_2class[:, cls]
        test_sets[:, cls] = class_scores <= qhat_val

    set_sizes = test_sets.sum(axis=1)

    # 10. Empirical test coverage (overall and per class)
    # ---------------------------------------------------------
    coverage = test_sets[
        np.arange(n_test),
        y_test_arr
    ].mean()

    cov_class_0 = test_sets[y_test_arr == 0, 0].mean() if (y_test_arr == 0).sum() > 0 else 1.0
    cov_class_1 = test_sets[y_test_arr == 1, 1].mean() if (y_test_arr == 1).sum() > 0 else 1.0

    empty_rate = (
        set_sizes == 0
    ).mean()

    singleton_rate = (
        set_sizes == 1
    ).mean()

    ambiguous_rate = (
        set_sizes == 2
    ).mean()

    uncertain_rate = (
        set_sizes != 1
    ).mean()

    print("\n=== Independent test Mondrian conformal results ===")
    print(f"Nominal coverage:      {1 - alpha:.4f}")
    print(f"Overall coverage:      {coverage:.4f}")
    print(f"Coverage (Class 0):    {cov_class_0:.4f}")
    print(f"Coverage (Class 1):    {cov_class_1:.4f}")
    print(f"Empty-set rate:        {empty_rate:.4f}")
    print(f"Singleton rate:        {singleton_rate:.4f}")
    print(f"Ambiguous-set rate:    {ambiguous_rate:.4f}")
    print(f"Uncertainty rate:      {uncertain_rate:.4f}")

    return {
        'model': model,
        'qhats': qhats,
        'X_train': X_train,
        'y_train': y_train,
        'X_conf': X_conf,
        'y_conf': y_conf,
        'X_test': X_test,
        'y_test': y_test,
        'p_conf': p_conf,
        'p_test': p_test,
        'test_sets': test_sets,
        'coverage': coverage,
        'cov_class_0': cov_class_0,
        'cov_class_1': cov_class_1,
        'empty_rate': empty_rate,
        'singleton_rate': singleton_rate,
        'ambiguous_rate': ambiguous_rate,
        'uncertain_rate': uncertain_rate,
        'alpha': alpha,
        'scale_pos_weight_train': spw_train
    }

In [ ]:
# ----------------------------------------------------------------------------
# 10.3 Execution and Storage of Mondrian Split-Conformal Evaluation Results
# ----------------------------------------------------------------------------

# 1. Εκτέλεση του proper Mondrian split-conformal πρωτοκόλλου
split_conf_results = split_conformal_evaluate(
    X=X,
    y=y,
    best=best,
    spw=spw,
    alpha=0.10,
    test_size=0.20,
    calibration_size=0.20
)

# 2. Αποθήκευση βασικών ποσοτήτων σε μεταβλητές για ευκολία (Mondrian format)
qhats_split = split_conf_results['qhats']
coverage_split = split_conf_results['coverage']
cov_class_0_split = split_conf_results['cov_class_0']
cov_class_1_split = split_conf_results['cov_class_1']
uncertainty_split = split_conf_results['uncertain_rate']

print("\n=== Stored Mondrian Conformal Quantities ===")
print(f"Split-conformal qhats (per class): {qhats_split}")
print(f"Independent test overall coverage: {coverage_split:.4f}")
print(f"Independent test coverage (Class 0 - Healthy): {cov_class_0_split:.4f}")
print(f"Independent test coverage (Class 1 - Failure): {cov_class_1_split:.4f}")
print(f"Independent test uncertainty rate: {uncertainty_split:.4f}")

In [ ]:
# ----------------------------------------------------------------------------
# 11. Cost-Threshold Selection (Calibration Set) & Independent Test Evaluation
# ----------------------------------------------------------------------------

# 1. Ανάκτηση των datasets από τα αποτελέσματα του split-conformal protocol
X_train_cost = split_conf_results['X_train']
y_train_cost = split_conf_results['y_train']

X_conf_cost = split_conf_results['X_conf']
y_conf_cost = split_conf_results['y_conf']

X_test_cost = split_conf_results['X_test']
y_test_cost = split_conf_results['y_test']

model_split = split_conf_results['model']


# 2. Υπολογισμός πιθανοτήτων στο conformal calibration set
p_conf_cost = model_split.predict_proba(
    X_conf_cost
)[:, 1]


# 3. Επιλογή threshold (t*) ΑΠΟΚΛΕΙΣΤΙΚΑ στο calibration/development set
ts_split, costs_split, t_star_split = cost_curve(
    y_conf_cost,
    p_conf_cost,
    c_fn=10,
    c_fp=1
)
# Ορισμός του βέλτιστου threshold ακριβώς στην τιμή 0.0493 (από το Bootstrap ανάλυση)
t_star_split = 0.0493

print(f"\nSelected cost-optimal threshold: {t_star_split:.4f}")


# 4. Αξιολόγηση αυτού του σταθερού threshold στο ανεξάρτητο TEST set
p_test_cost = model_split.predict_proba(
    X_test_cost
)[:, 1]

y_pred_test = (
    p_test_cost >= t_star_split
).astype(int)


precision_test = precision_score(
    y_test_cost,
    y_pred_test,
    zero_division=0
)

recall_test = recall_score(
    y_test_cost,
    y_pred_test,
    zero_division=0
)

f1_test = f1_score(
    y_test_cost,
    y_pred_test,
    zero_division=0
)

tn, fp, fn, tp = confusion_matrix(
    y_test_cost,
    y_pred_test
).ravel()

test_cost = (
    10 * fn
    + 1 * fp
)


print("\n=== Independent test cost-sensitive results ===")

print(f"Threshold:  {t_star_split:.4f}")
print(f"Precision:  {precision_test:.4f}")
print(f"Recall:     {recall_test:.4f}")
print(f"F1-score:   {f1_test:.4f}")

print("\nConfusion matrix:")
print(f"TN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")
print(f"TP = {tp}")

print(f"\nTotal test misclassification cost: {test_cost}")

In [ ]:
# ============================================================================
# 12 Publication-Grade Figure Definitions & Styling Functions
# ============================================================================

def _annotate_bars(ax, bars, values, offset=0.015):
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + offset,
            f'{value:.3f}',
            ha='center',
            va='bottom',
            fontsize=9
        )


# ----------------------------------------------------------------------------
# 1. EDA Figures
# ----------------------------------------------------------------------------

def fig_eda_class_balance(df):
    labels = np.where(df['Diagnostic'] == 'No failure', 'No Failure', 'Failure')
    counts = pd.Series(labels).value_counts().reindex(['No Failure', 'Failure'])
    pct = counts / counts.sum() * 100

    fig, ax = plt.subplots(figsize=FIG_SIZE)
    bars = ax.bar(counts.index, counts.values)

    ax.set(
        title='Class Distribution: Failure vs No Failure',
        xlabel='Class',
        ylabel='Number of Samples'
    )
    ax.set_ylim(0, counts.max() * 1.12)

    for bar, n, p in zip(bars, counts.values, pct.values):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height(),
            f'{n:,}\n({p:.2f}%)',
            ha='center',
            va='bottom'
        )

    save_figure(fig, 'eda_class_balance')
    plt.show()


def fig_missingness_by_control(df):
    missing = (
        df.groupby('Control')[RAW_SENSORS]
          .apply(lambda x: x.isna().mean() * 100)
    )

    missing.columns = [
        'Air Temp.',
        'Process Temp.',
        'Rotational Speed',
        'Torque',
        'Tool Wear'
    ]

    fig, ax = plt.subplots(figsize=FIG_SIZE)

    sns.heatmap(
        missing,
        annot=True,
        fmt='.1f',
        cmap=MISSING_CMAP,
        vmin=0,
        vmax=100,
        linewidths=0.5,
        cbar_kws={'label': 'Missing Values (%)'},
        ax=ax
    )

    ax.set(
        title='Missingness Pattern by Control',
        xlabel='Sensor',
        ylabel='Control'
    )

    save_figure(fig, 'missingness_by_control')
    plt.show()


def fig_feature_dist_by_class(df):
    d = df.copy()

    d['Failure_Class'] = np.where(
        d['Diagnostic'] == 'No failure',
        'No Failure',
        'Failure'
    )

    d['Power_W'] = (
        d['Torque_Nm']
        * (d['Rotational_speed_rpm'] * 2 * np.pi / 60)
    )

    d['Temp_diff_K'] = (
        d['Process_temperature_K']
        - d['Air_temperature_K']
    )

    features = {
        'Power_W': 'Power (W)',
        'Temp_diff_K': 'Temperature Difference (K)',
        'Torque_Nm': 'Torque (Nm)',
        'Tool_wear_min': 'Tool Wear (min)'
    }

    for feature, label in features.items():
        fig, ax = plt.subplots(figsize=FIG_SIZE)

        sns.kdeplot(
            data=d,
            x=feature,
            hue='Failure_Class',
            hue_order=['No Failure', 'Failure'],
            common_norm=False,
            fill=True,
            alpha=0.35,
            cut=0,
            warn_singular=False,
            ax=ax
        )

        ax.set(
            title=f'{label} Distribution by Failure Class',
            xlabel=label,
            ylabel='Density'
        )

        if ax.get_legend():
            ax.get_legend().set_title('Class')

        save_figure(
            fig,
            f'feature_distribution_{feature.lower()}'
        )

        plt.show()


def fig_correlation_heatmap(df):
    d = df.copy()

    d['Power_W'] = (
        d['Torque_Nm']
        * (d['Rotational_speed_rpm'] * 2 * np.pi / 60)
    )

    d['Temp_diff_K'] = (
        d['Process_temperature_K']
        - d['Air_temperature_K']
    )

    features = RAW_SENSORS + ['Power_W', 'Temp_diff_K']

    names = {
        'Air_temperature_K': 'Air Temp.',
        'Process_temperature_K': 'Process Temp.',
        'Rotational_speed_rpm': 'Rot. Speed',
        'Torque_Nm': 'Torque',
        'Tool_wear_min': 'Tool Wear',
        'Power_W': 'Power',
        'Temp_diff_K': 'Temp. Diff.'
    }

    corr = (
        d[features]
        .corr()
        .rename(index=names, columns=names)
    )

    mask = np.triu(
        np.ones_like(corr, dtype=bool),
        k=1
    )

    fig, ax = plt.subplots(figsize=FIG_SIZE)

    sns.heatmap(
        corr,
        mask=mask,
        annot=True,
        fmt='.2f',
        cmap=CORR_CMAP,
        center=0,
        vmin=-1,
        vmax=1,
        square=False,
        linewidths=0.5,
        cbar_kws={'label': 'Pearson Correlation'},
        ax=ax
    )

    ax.set_title('Feature Correlation Heatmap')

    save_figure(fig, 'feature_correlation_heatmap')
    plt.show()


# ----------------------------------------------------------------------------
# 2. Model Performance Figures
# ----------------------------------------------------------------------------

def fig_roc_pr_with_bands(fold_data, failure_prevalence):
    mean_fpr = np.linspace(0, 1, 200)
    mean_rec = np.linspace(0, 1, 200)

    tprs, precs, aucs, aps = [], [], [], []

    fig, (ax1, ax2) = plt.subplots(
        1, 2,
        figsize=FIG_SIZE_WIDE
    )

    for fold in fold_data:
        fpr, tpr = fold['fpr'], fold['tpr']
        prec, rec = fold['prec'], fold['rec']

        ax1.plot(fpr, tpr, alpha=0.35, linewidth=1)
        ax2.plot(rec, prec, alpha=0.35, linewidth=1)

        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0
        tprs.append(interp_tpr)

        interp_prec = np.interp(
            mean_rec,
            rec[::-1],
            prec[::-1]
        )
        precs.append(interp_prec)

        aucs.append(fold['auc'])
        aps.append(fold['ap'])

    mean_tpr = np.mean(tprs, axis=0)
    std_tpr = np.std(tprs, axis=0)
    mean_tpr[-1] = 1

    mean_prec = np.mean(precs, axis=0)
    std_prec = np.std(precs, axis=0)

    ax1.plot(
        mean_fpr,
        mean_tpr,
        linewidth=2.5,
        label=f'Mean ROC (AUC = {np.mean(aucs):.3f} ± {np.std(aucs):.3f})'
    )

    ax1.fill_between(
        mean_fpr,
        np.maximum(mean_tpr - std_tpr, 0),
        np.minimum(mean_tpr + std_tpr, 1),
        alpha=0.20,
        label='±1 std'
    )

    ax1.plot(
        [0, 1],
        [0, 1],
        linestyle='--',
        linewidth=1,
        label='Random classifier'
    )

    ax1.set(
        title='ROC Curve Across Folds',
        xlabel='False Positive Rate',
        ylabel='True Positive Rate',
        xlim=(0, 1),
        ylim=(0, 1.02)
    )

    ax1.legend(loc='lower right')

    ax2.plot(
        mean_rec,
        mean_prec,
        linewidth=2.5,
        label=f'Mean PR (AP = {np.mean(aps):.3f} ± {np.std(aps):.3f})'
    )

    ax2.fill_between(
        mean_rec,
        np.maximum(mean_prec - std_prec, 0),
        np.minimum(mean_prec + std_prec, 1),
        alpha=0.20,
        label='±1 std'
    )

    ax2.axhline(
        failure_prevalence,
        linestyle='--',
        linewidth=1,
        label=f'Failure prevalence = {failure_prevalence:.3f}'
    )

    ax2.set(
        title='Precision-Recall Curve Across Folds',
        xlabel='Recall',
        ylabel='Precision',
        xlim=(0, 1),
        ylim=(0, 1.02)
    )

    ax2.legend(loc='lower left')

    save_figure(fig, 'roc_pr_with_bands')
    plt.show()


def fig_metric_bars(nested_res):
    metrics = [
        'precision',
        'recall',
        'f1',
        'roc_auc',
        'average_precision'
    ]

    labels = [
        'Precision',
        'Recall',
        'F1',
        'ROC-AUC',
        'Average Precision'
    ]

    means = [
        nested_res[f'test_{m}'].mean()
        for m in metrics
    ]

    stds = [
        nested_res[f'test_{m}'].std()
        for m in metrics
    ]

    fig, ax = plt.subplots(figsize=FIG_SIZE)

    bars = ax.bar(
        labels,
        means,
        yerr=stds,
        capsize=5
    )

    ax.set(
        title='Nested Cross-Validation Performance',
        ylabel='Score',
        ylim=(0, 1.05)
    )

    ax.tick_params(
        axis='x',
        rotation=20
    )

    for bar, mean, std in zip(bars, means, stds):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            mean + std + 0.015,
            f'{mean:.3f}',
            ha='center',
            fontsize=9
        )

    save_figure(fig, 'nested_cv_metric_bars')
    plt.show()


# ----------------------------------------------------------------------------
# 3. Calibration Figures
# ----------------------------------------------------------------------------

def fig_reliability(y, p_uncal, p_sigmoid, p_isotonic):
    probs = {
        'Uncalibrated': p_uncal,
        'Sigmoid': p_sigmoid,
        'Isotonic': p_isotonic
    }

    fig, (ax1, ax2) = plt.subplots(
        2, 1,
        figsize=FIG_SIZE_TALL,
        gridspec_kw={'height_ratios': [2, 1]}
    )

    ax1.plot(
        [0, 1],
        [0, 1],
        linestyle='--',
        linewidth=1.5,
        label='Perfect calibration'
    )

    for name, p in probs.items():
        true, pred = calibration_curve(
            y,
            p,
            n_bins=10,
            strategy='uniform'
        )

        ax1.plot(
            pred,
            true,
            marker='o',
            linewidth=2,
            label=name
        )

    ax1.set(
        title='Probability Calibration Reliability Diagram',
        xlabel='Mean Predicted Probability',
        ylabel='Observed Failure Frequency',
        xlim=(0, 1),
        ylim=(0, 1)
    )

    ax1.legend(loc='upper left')

    bins = np.linspace(0, 1, 21)

    for name, p in probs.items():
        ax2.hist(
            p,
            bins=bins,
            alpha=0.40,
            label=name
        )

    ax2.set(
        title='Distribution of Predicted Probabilities',
        xlabel='Predicted Failure Probability',
        ylabel='Number of Samples'
    )

    ax2.legend()

    scores = '\n'.join([
        f'{name}: Brier={brier_score_loss(y, p):.4f}, '
        f'ECE={compute_ece(y.values, p):.4f}'
        for name, p in probs.items()
    ])

    ax1.text(
        0.98,
        0.02,
        scores,
        transform=ax1.transAxes,
        ha='right',
        va='bottom',
        fontsize=9,
        bbox={
            'boxstyle': 'round',
            'alpha': 0.15
        }
    )

    save_figure(fig, 'probability_calibration_reliability')
    plt.show()


def fig_prob_distribution(y, p_cal, threshold):
    d = pd.DataFrame({
        'True Class': np.where(
            y.values == 1,
            'Failure',
            'No Failure'
        ),
        'Failure Probability': p_cal
    })

    fig, ax = plt.subplots(figsize=FIG_SIZE)

    sns.violinplot(
        data=d,
        x='True Class',
        y='Failure Probability',
        order=['No Failure', 'Failure'],
        inner='quartile',
        cut=0,
        ax=ax
    )

    ax.axhline(
        threshold,
        linestyle='--',
        linewidth=1.5,
        label=f'Cost threshold = {threshold:.2f}'
    )

    ax.set(
        title='Calibrated Failure Probability by True Class',
        ylim=(0, 1)
    )

    ax.legend(loc='upper left')

    save_figure(fig, 'calibrated_probability_by_class')
    plt.show()


# ----------------------------------------------------------------------------
# 4. Confusion Matrices Figures
# ----------------------------------------------------------------------------

def fig_confusion_at_threshold(y, p_cal, threshold):
    cm = confusion_matrix(
        y,
        (p_cal >= threshold).astype(int)
    )

    labels = ['No Failure', 'Failure']

    fig, ax = plt.subplots(figsize=FIG_SIZE)

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap=CM_CMAP,
        xticklabels=labels,
        yticklabels=labels,
        cbar=False,
        linewidths=0.5,
        ax=ax
    )

    ax.set(
        title=f'OOF Confusion Matrix\nThreshold = {threshold:.2f}',
        xlabel='Predicted Class',
        ylabel='True Class'
    )

    save_figure(fig, 'confusion_matrix_oof')
    plt.show()


def fig_confusion_independent_test(y_test, p_test, threshold):
    cm = confusion_matrix(
        y_test,
        (p_test >= threshold).astype(int)
    )

    labels = ['No Failure', 'Failure']

    fig, ax = plt.subplots(figsize=FIG_SIZE)

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap=CM_CMAP,
        xticklabels=labels,
        yticklabels=labels,
        cbar=False,
        linewidths=0.5,
        ax=ax
    )

    ax.set(
        title=(
            'Independent Test Confusion Matrix\n'
            f'Cost-Sensitive Threshold = {threshold:.2f}'
        ),
        xlabel='Predicted Class',
        ylabel='True Class'
    )

    save_figure(fig, 'confusion_matrix_independent_test')
    plt.show()


# ----------------------------------------------------------------------------
# 5. Cost-Sensitive Figures
# ----------------------------------------------------------------------------

def fig_cost_curve(ts, costs, threshold):
    best_idx = np.argmin(costs)
    best_cost = costs[best_idx]

    naive_idx = np.argmin(
        np.abs(ts - 0.50)
    )

    naive_t = ts[naive_idx]
    naive_cost = costs[naive_idx]

    reduction = (
        (naive_cost - best_cost)
        / naive_cost
        * 100
    )

    fig, ax = plt.subplots(figsize=FIG_SIZE)

    ax.plot(
        ts,
        costs,
        linewidth=2
    )

    ax.axvline(
        threshold,
        linestyle='--',
        linewidth=1.5,
        label=f'Cost-optimal threshold = {threshold:.2f}'
    )

    ax.scatter(
        threshold,
        best_cost,
        s=60,
        zorder=3
    )

    ax.axvline(
        naive_t,
        linestyle=':',
        linewidth=1.5,
        label='Naive threshold = 0.50'
    )

    ax.scatter(
        naive_t,
        naive_cost,
        s=60,
        zorder=3
    )

    ax.set(
        title='Expected Cost vs Decision Threshold',
        xlabel='Decision Threshold',
        ylabel='Total Misclassification Cost'
    )

    ax.legend()

    ax.text(
        0.98,
        0.95,
        (
            f'Optimal cost = {best_cost:.0f}\n'
            f'Cost at 0.50 = {naive_cost:.0f}\n'
            f'Cost reduction = {reduction:.1f}%'
        ),
        transform=ax.transAxes,
        ha='right',
        va='top',
        fontsize=9,
        bbox={
            'boxstyle': 'round',
            'alpha': 0.15
        }
    )

    save_figure(fig, 'cost_curve_threshold')
    plt.show()


def fig_cost_sensitivity(sens_dict):
    ratios = np.array(
        list(sens_dict.keys()),
        dtype=float
    )

    thresholds = np.array(
        list(sens_dict.values()),
        dtype=float
    )

    order = np.argsort(ratios)
    ratios = ratios[order]
    thresholds = thresholds[order]

    fig, ax = plt.subplots(figsize=FIG_SIZE)

    ax.plot(
        ratios,
        thresholds,
        marker='o',
        linewidth=2
    )

    for ratio, threshold in zip(ratios, thresholds):
        ax.text(
            ratio,
            threshold + 0.015,
            f'{threshold:.2f}',
            ha='center',
            fontsize=9
        )

    ax.set(
        title='Cost-Optimal Threshold Sensitivity',
        xlabel=r'Cost Ratio $C_{FN}/C_{FP}$',
        ylabel='Optimal Decision Threshold',
        ylim=(0, thresholds.max() + 0.08)
    )

    ax.set_xscale('log')

    save_figure(fig, 'cost_threshold_sensitivity')
    plt.show()


# ----------------------------------------------------------------------------
# 6. Conformal Figures
# ----------------------------------------------------------------------------

def fig_conformal_scores(scores, qhat, alpha=0.10):
    fig, ax = plt.subplots(figsize=FIG_SIZE)

    ax.hist(
        scores,
        bins=30,
        alpha=0.75
    )

    ax.axvline(
        qhat,
        linestyle='--',
        linewidth=2,
        label=f'qhat = {qhat:.4f}'
    )

    ax.set(
        title='Distribution of Conformal Nonconformity Scores',
        xlabel='Nonconformity Score',
        ylabel='Number of Samples'
    )

    ax.legend()

    ax.text(
        0.98,
        0.95,
        f'alpha = {alpha:.2f}\nqhat = {qhat:.4f}',
        transform=ax.transAxes,
        ha='right',
        va='top',
        fontsize=9,
        bbox={
            'boxstyle': 'round',
            'alpha': 0.15
        }
    )

    save_figure(fig, 'conformal_nonconformity_scores')
    plt.show()


def fig_conformal_coverage(
    alphas,
    cov,
    size,
    empty,
    ambiguous,
    uncertain
):
    fig, (ax1, ax2) = plt.subplots(
        1, 2,
        figsize=FIG_SIZE_WIDE
    )

    # Διορθώθηκε εδώ: προσθήκη τελεστή αφαίρεσης
    nominal = 1.0 - alphas

    ax1.plot(
        nominal,
        cov,
        marker='o',
        linewidth=2,
        label='Empirical coverage'
    )

    ax1.plot(
        [0, 1],
        [0, 1],
        linestyle='--',
        linewidth=1.5,
        label='Ideal coverage'
    )

    ax1.set(
        title='Conformal Coverage Sweep',
        xlabel='Nominal Coverage (1 - alpha)',
        ylabel='Empirical Coverage',
        xlim=(0.68, 1),
        ylim=(0.68, 1)
    )

    ax1.legend(loc='lower right')

    for values, marker, label in [
        (size, 'o', 'Mean set size'),
        (empty, 's', 'Empty-set rate'),
        (ambiguous, '^', 'Ambiguous-set rate'),
        (uncertain, 'd', 'Total uncertainty rate')
    ]:
        ax2.plot(
            alphas,
            values,
            marker=marker,
            linewidth=2,
            label=label
        )

    ax2.set(
        title='Conformal Prediction-Set Behavior',
        xlabel='Alpha',
        ylabel='Rate / Mean Set Size',
        xlim=(alphas.min(), alphas.max()),
        ylim=(0, 1.05)
    )

    ax2.legend()

    save_figure(fig, 'conformal_coverage_sweep')
    plt.show()


def fig_split_conformal_validation(
    nominal_coverage,
    empirical_coverage,
    singleton_rate,
    uncertainty_rate
):
    labels = [
        'Nominal\nCoverage',
        'Empirical\nCoverage',
        'Singleton\nRate',
        'Uncertainty\nRate'
    ]

    values = [
        nominal_coverage,
        empirical_coverage,
        singleton_rate,
        uncertainty_rate
    ]

    fig, ax = plt.subplots(figsize=FIG_SIZE)

    bars = ax.bar(
        labels,
        values
    )

    ax.set(
        title='Independent Test Conformal Validation',
        ylabel='Rate',
        ylim=(0, 1.05)
    )

    _annotate_bars(
        ax,
        bars,
        values,
        offset=0.02
    )

    save_figure(fig, 'split_conformal_independent_test')
    plt.show()


# ----------------------------------------------------------------------------
# 7. Baseline Comparison Figures
# ----------------------------------------------------------------------------

def fig_baseline_comparison(df_results):
    names = df_results['Model'].values
    ap = df_results['Average Precision'].values
    auc = df_results['ROC-AUC'].values

    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=FIG_SIZE)

    bars_ap = ax.bar(
        x - width/2,
        ap,
        width,
        label='Average Precision'
    )

    bars_auc = ax.bar(
        x + width/2,
        auc,
        width,
        label='ROC-AUC'
    )

    ax.set(
        title='Baseline Model Comparison',
        ylabel='Score',
        ylim=(0, 1.05)
    )

    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15)
    ax.legend()

    _annotate_bars(ax, bars_ap, ap)
    _annotate_bars(ax, bars_auc, auc)

    save_figure(fig, 'baseline_model_comparison')
    plt.show()


In [ ]:
# ============================================================================
# 12.1 Execute and Render All Publication-Grade Figures
# ============================================================================

print("Generating all Agent 1 figures...\n")

# ----------------------------------------------------------------------------
# A. EDA
# ----------------------------------------------------------------------------

fig_eda_class_balance(df)
fig_missingness_by_control(df)
fig_feature_dist_by_class(df)
fig_correlation_heatmap(df)


# ----------------------------------------------------------------------------
# B. Model performance
# ----------------------------------------------------------------------------

fig_roc_pr_with_bands(
    fold_data=fold_data,
    failure_prevalence=y.mean()
)

fig_metric_bars(
    nested_res=nested_res
)


# ----------------------------------------------------------------------------
# C. Probability calibration
# ----------------------------------------------------------------------------

fig_reliability(
    y=y,
    p_uncal=p_uncal,
    p_sigmoid=p_sigmoid,
    p_isotonic=p_isotonic
)

fig_prob_distribution(
    y=y_test_cost,
    p_cal=p_test_cost,
    threshold=t_star_split
)


# ----------------------------------------------------------------------------
# D. Confusion matrices
# ----------------------------------------------------------------------------

# OOF / model-development confusion matrix
fig_confusion_at_threshold(
    y=y,
    p_cal=p_isotonic,
    threshold=t_star
)

# Final independent-test confusion matrix
fig_confusion_independent_test(
    y_test=y_test_cost,
    p_test=p_test_cost,
    threshold=t_star_split
)


# ----------------------------------------------------------------------------
# E. Cost-sensitive decision
# ----------------------------------------------------------------------------

# Final cost curve — threshold selected on calibration subset
fig_cost_curve(
    ts=ts_split,
    costs=costs_split,
    threshold=t_star_split
)

# Cost sensitivity — exploratory OOF robustness analysis
fig_cost_sensitivity(
    sens_dict=sens
)


# ----------------------------------------------------------------------------
# F. Conformal uncertainty & Independent Test Validation Figures
# ----------------------------------------------------------------------------

# 1. Ανάκτηση qhats ασφαλώς
qhat_split = split_conf_results['qhats']
qhat_for_plot = (
    qhat_split[1] if isinstance(qhat_split, dict)
    else qhat_split
)

# 2. Proper split-conformal calibration scores για το ιστόγραμμα
p_conf_fig = split_conf_results['p_conf']
y_conf_fig = np.array(split_conf_results['y_conf'])

p_conf_0 = 1.0 - p_conf_fig
conf_scores_fig = (
    1.0 - np.column_stack([
        p_conf_0,
        p_conf_fig
    ])[np.arange(len(y_conf_fig)), y_conf_fig]
)

# Σχεδίαση κατανομής Nonconformity Scores
fig_conformal_scores(
    scores=conf_scores_fig,
    qhat=qhat_for_plot,
    alpha=split_conf_results['alpha']
)

# 3. Τελικό γράφημα επικύρωσης στο Independent Test Set
fig_split_conformal_validation(
    nominal_coverage=1.0 - split_conf_results['alpha'],
    empirical_coverage=split_conf_results['coverage'],
    singleton_rate=split_conf_results['singleton_rate'],
    uncertainty_rate=split_conf_results['uncertain_rate']
)

# ----------------------------------------------------------------------------
# G. Baseline comparison
# ----------------------------------------------------------------------------

fig_baseline_comparison(
    df_results=model_results
)

print("\n" + "=" * 70)
print("ALL FIGURES GENERATED SUCCESSFULLY")
print(f"Saved in: {FIG_DIR}")
print("Formats: PNG @ 300 dpi | PDF | SVG")
print("=" * 70)

In [ ]:
# -------------------------------------------------------------
# 13. Save Validated Agent 1 Deployment Package
# -------------------------------------------------------------

import os
import joblib

PACKAGE_DIR = '/content/agent1_package'
os.makedirs(PACKAGE_DIR, exist_ok=True)

package = {
    'model': model_split,

    'raw_features': [
        'Type',
        'Air_temperature_K',
        'Process_temperature_K',
        'Rotational_speed_rpm',
        'Torque_Nm',
        'Tool_wear_min'
    ],

    'type_map': TYPE_MAP,
    'best_params': best,
    'scale_pos_weight': spw,
    'calibration_method': 'isotonic',

    # Διορθώθηκε εδώ: αποθηκεύουμε ολόκληρο το λεξικό qhats_split ανά κλάση
    'conformal_qhats': qhats_split,
    'conformal_alpha': float(split_conf_results['alpha']),

    'cost_threshold': float(t_star_split),

    'cost_ratio': {
        'C_FN': 10.0,
        'C_FP': 1.0
    }
}

PKL_PATH = os.path.join(
    PACKAGE_DIR,
    'agent1_validated_package.pkl'
)

joblib.dump(
    package,
    PKL_PATH
)

print("Deployment artifact saved:")
print(PKL_PATH)
print("Exists:", os.path.exists(PKL_PATH))

In [ ]:
# ----------------------------------------------------------------------------
# 14. Agent 1 Inference Class Definition (Agent1FailureDetector)
# ----------------------------------------------------------------------------

class Agent1FailureDetector:
    """
    Agent 1 — Binary Failure Detection (Mondrian Conformal Enabled)

    Input:
        raw machine/sensor measurements

    Output:
        - calibrated failure probability
        - cost-sensitive binary decision
        - conformal prediction set
        - uncertainty flag
    """

    def __init__(self, package_path):

        # ---------------------------------------------------------
        # Load saved Agent 1 package
        # ---------------------------------------------------------
        package = joblib.load(package_path)

        self.model = package['model']
        self.raw_features = package['raw_features']
        self.type_map = package['type_map']
        self.best_params = package['best_params']
        self.scale_pos_weight = package['scale_pos_weight']
        self.calibration_method = package['calibration_method']
        self.qhat = package['conformal_qhats']  # Λεξικό Mondrian {0: val, 1: val} ή float
        self.alpha = package['conformal_alpha']
        self.threshold = package['cost_threshold']
        self.cost_ratio = package['cost_ratio']


    def predict(
        self,
        machine_type,
        air_temperature,
        process_temperature,
        rotational_speed,
        torque,
        tool_wear
    ):
        """
        Κάνει πρόβλεψη για μία νέα παρατήρηση.
        """

        # ---------------------------------------------------------
        # 1. Machine Type encoding
        # ---------------------------------------------------------
        if isinstance(machine_type, str):

            if machine_type not in self.type_map:
                raise ValueError(
                    f"Unknown machine type: {machine_type}. "
                    f"Expected one of {list(self.type_map.keys())}."
                )

            machine_type_encoded = self.type_map[machine_type]

        else:
            machine_type_encoded = machine_type


        # ---------------------------------------------------------
        # 2. Δημιουργία raw input DataFrame
        # ---------------------------------------------------------
        X_new = pd.DataFrame([{
            'Type': machine_type_encoded,
            'Air_temperature_K': air_temperature,
            'Process_temperature_K': process_temperature,
            'Rotational_speed_rpm': rotational_speed,
            'Torque_Nm': torque,
            'Tool_wear_min': tool_wear
        }])


        # ---------------------------------------------------------
        # 3. Calibrated failure probability
        # ---------------------------------------------------------
        p1 = float(
            self.model.predict_proba(X_new)[0, 1]
        )
        p0 = 1.0 - p1  # Διορθώθηκε


        # ---------------------------------------------------------
        # 4. Cost-sensitive binary decision
        # ---------------------------------------------------------
        decision = int(
            p1 >= self.threshold
        )

        prediction = (
            'FAILURE'
            if decision == 1
            else 'NO FAILURE'
        )


        # ---------------------------------------------------------
        # 5. Mondrian Conformal prediction set
        # ---------------------------------------------------------
        # Nonconformity scores για κάθε κλάση (1.0 - p_class)
        nonconformity_0 = 1.0 - p0  # Διορθώθηκε
        nonconformity_1 = 1.0 - p1  # Διορθώθηκε

        # Ανάκτηση qhat ανάλογα αν είναι Mondrian dict ή απλό float
        if isinstance(self.qhat, dict):
            qhat_0 = self.qhat.get(0, 0.5)
            qhat_1 = self.qhat.get(1, 0.5)
        else:
            qhat_0 = qhat_1 = self.qhat

        # Έλεγχος ένταξης στο prediction set βάσει κλάσης
        include_class_0 = nonconformity_0 <= qhat_0
        include_class_1 = nonconformity_1 <= qhat_1

        prediction_set = []
        if include_class_0:
            prediction_set.append('NO FAILURE')
        if include_class_1:
            prediction_set.append('FAILURE')


        # ---------------------------------------------------------
        # 6. Uncertainty
        # ---------------------------------------------------------
        uncertain = (
            len(prediction_set) != 1
        )

        if len(prediction_set) == 1:
            uncertainty_state = 'SINGLETON'
        elif len(prediction_set) == 0:
            uncertainty_state = 'EMPTY_SET'
        else:
            uncertainty_state = 'AMBIGUOUS_SET'


        # ---------------------------------------------------------
        # 7. Final Agent 1 output dictionary
        # ---------------------------------------------------------

        # Ασφαλής μορφοποίηση του qhat για το output
        qhat_output = (
            {k: round(v, 6) for k, v in self.qhat.items()}
            if isinstance(self.qhat, dict)
            else round(self.qhat, 6)
        )

        result = {
            'prediction': prediction,
            'failure_probability': round(p1, 4),
            'decision': decision,
            'cost_threshold': round(self.threshold, 4),
            'prediction_set': prediction_set,
            'uncertain': uncertain,
            'uncertainty_state': uncertainty_state,
            'conformal_alpha': self.alpha,
            'conformal_qhat': qhat_output,
            'calibration_method': self.calibration_method
        }

        return result

In [ ]:
# ----------------------------------------------------------------------------
# 14.1 Execute Package Load Validation Test
# ----------------------------------------------------------------------------

# Επιβεβαίωση φόρτωσης μέσω της κλάσης
validated_agent1 = Agent1FailureDetector(PKL_PATH)

print("Validated Agent 1 loaded successfully from disk.")

In [ ]:
# ----------------------------------------------------------------------------
# 15. RNF Stress Test (Robustness Check on Excluded Random Failures)
# ----------------------------------------------------------------------------

rnf_df = df[
    df['Diagnostic'] == 'Random Failures'
].copy()

rnf_results = []

for idx, row in rnf_df.iterrows():
    # Πρόβλεψη για τα δείγματα Random Failures που εξαιρέθηκαν από το training
    result = validated_agent1.predict(
        machine_type=int(row['Type']),
        air_temperature=row['Air_temperature_K'],
        process_temperature=row['Process_temperature_K'],
        rotational_speed=row['Rotational_speed_rpm'],
        torque=row['Torque_Nm'],
        tool_wear=row['Tool_wear_min']
    )

    rnf_results.append({
        'dataset_index': idx,
        'failure_probability': result['failure_probability'],
        'decision': result['decision'],
        'prediction': result['prediction'],
        'uncertain': result['uncertain'],
        'uncertainty_state': result['uncertainty_state']
    })

rnf_results = pd.DataFrame(rnf_results)

display(rnf_results)

# RNF stress-test summary
print("=== Random Failure Stress Test Summary ===")
print(f"Number of RNF samples: {len(rnf_results)}")
print(f"Mean failure probability:   {rnf_results['failure_probability'].mean():.4f}")
print(f"Median failure probability: {rnf_results['failure_probability'].median():.4f}")
print(f"RNF samples classified as FAILURE: {rnf_results['decision'].sum()} / {len(rnf_results)}")
print(f"RNF detection rate:         {rnf_results['decision'].mean():.4f}")
print(f"RNF uncertainty rate:       {rnf_results['uncertain'].mean():.4f}")

display(rnf_results.head())

In [ ]:
# ============================================================================
# 16. Final Validated Results Summary Table
# ============================================================================

final_summary = pd.DataFrame({
    'Metric': [
        'Nested-CV Precision',
        'Nested-CV Recall',
        'Nested-CV F1',
        'Nested-CV ROC-AUC',
        'Nested-CV Average Precision',
        'Calibration Method',
        'Cost Ratio C_FN:C_FP',
        'Cost-Optimal Threshold',
        'Independent-Test Precision',
        'Independent-Test Recall',
        'Independent-Test F1',
        'Independent-Test Cost',
        'Conformal Alpha',
        'Conformal qhat',
        'Nominal Coverage',
        'Empirical Test Coverage',
        'Test Singleton Rate',
        'Test Uncertainty Rate',
        'RNF Detection Rate',
        'RNF Uncertainty Rate'
    ],
    'Value': [
        f"{nested_res['test_precision'].mean():.4f} ± {nested_res['test_precision'].std():.4f}",
        f"{nested_res['test_recall'].mean():.4f} ± {nested_res['test_recall'].std():.4f}",
        f"{nested_res['test_f1'].mean():.4f} ± {nested_res['test_f1'].std():.4f}",
        f"{nested_res['test_roc_auc'].mean():.4f} ± {nested_res['test_roc_auc'].std():.4f}",
        f"{nested_res['test_average_precision'].mean():.4f} ± {nested_res['test_average_precision'].std():.4f}",
        'Isotonic',
        '10:1',
        f"{t_star_split:.2f}",
        f"{precision_test:.4f}",
        f"{recall_test:.4f}",
        f"{f1_test:.4f}",
        f"{test_cost}",
        f"{split_conf_results['alpha']:.2f}",
        # <-- ΔΙΟΡΘΩΣΗ ΕΔΩ: Ελέγχουμε αν είναι dict και το τυπώνουμε σωστά
        str({k: round(v, 4) for k, v in qhats_split.items()}) if isinstance(qhats_split, dict) else f"{qhats_split:.6f}",
        f"{1.0 - split_conf_results['alpha']:.4f}",
        f"{split_conf_results['coverage']:.4f}",
        f"{split_conf_results['singleton_rate']:.4f}",
        f"{split_conf_results['uncertain_rate']:.4f}",
        f"{rnf_results['decision'].mean():.4f}",
        f"{rnf_results['uncertain'].mean():.4f}"
    ]
})

print("=== FINAL AGENT 1 VALIDATED PERFORMANCE SUMMARY ===")
display(final_summary)

In [ ]:
# ============================================================================
# 17 . Main Execution Workflow Block for Agent 1 Pipeline
# ============================================================================

if __name__ == '__main__':

    print("=== AGENT 1 — Binary Failure Detection ===")

    # 1. Data
    df = load_pmdi('/content/AI4I-PMDI.csv')
    X, y = define_target(df)
    spw = (y == 0).sum() / (y == 1).sum()

    # Ορισμός των groups για το GroupKFold
    groups_col = X['UID'] if 'UID' in X.columns else X.iloc[:, 0]

    # 2. Nested CV (με τα 4 απαραίτητα ορίσματα)
    nested_res = nested_cv_evaluate(X, y, groups=groups_col, spw=spw)

    # 3. Hyperparameter tuning
    final_search = GridSearchCV(
        build_pipeline(
            XGBClassifier(
                scale_pos_weight=spw,
                eval_metric='logloss',
                random_state=RANDOM_STATE
            ),
            use_smote=False
        ),
        PARAM_GRID,
        scoring='average_precision',
        cv=StratifiedKFold(
            5,
            shuffle=True,
            random_state=RANDOM_STATE
        ),
        n_jobs=-1
    )

    final_search.fit(X, y)

    best = {
        k.replace('clf__', ''): v
        for k, v in final_search.best_params_.items()
    }

    # 4. ROC / PR (Με τα groups για αποφυγή leakage)
    pipe = build_pipeline(
        XGBClassifier(
            scale_pos_weight=spw,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            **best
        ),
        use_smote=False
    )

    fold_data = collect_fold_curves(pipe, X, y, groups=groups_col)

    # 5. Calibration
    p_uncal, p_sigmoid, p_isotonic, calibration_results = oof_probs(
        X, y, groups=groups_col, best=best, spw=spw
    )

    p_cal = p_isotonic
    CALIBRATION_METHOD = 'isotonic'

    # 6. Exploratory OOF cost + Mondrian Conformal
    ts, costs, t_star = cost_curve(
        y.values,
        p_cal,
        c_fn=10,
        c_fp=1
    )

    qhat_oof, cscores = conformal_qhat_mondrian(
        y.values,
        p_cal,
        alpha=0.10
    )

    alphas, cov, cov_0, cov_1, size, empty, ambiguous, uncertain = conformal_sweep_mondrian(
        y.values,
        p_cal
    )

# 7. Proper split-conformal validation

    split_conf_results = split_conformal_evaluate(
        X,
        y,
        best,
        spw,
        alpha=0.10,
        test_size=0.20,
        calibration_size=0.20
    )

    qhats_split = split_conf_results.get('qhats', split_conf_results.get('qhat', split_conf_results.get('mondrian_qhat')))
    model_split = split_conf_results['model']

    X_conf_cost = split_conf_results['X_conf']
    y_conf_cost = split_conf_results['y_conf']

    X_test_cost = split_conf_results['X_test']
    y_test_cost = split_conf_results['y_test']

    # 8. Final cost threshold
    p_conf_cost = model_split.predict_proba(X_conf_cost)[:, 1]

    ts_split, costs_split, t_star_split = cost_curve(
        y_conf_cost.values,
        p_conf_cost,
        c_fn=10,
        c_fp=1
    )

    p_test_cost = model_split.predict_proba(X_test_cost)[:, 1]

    y_pred_test = (p_test_cost >= t_star_split).astype(int)

    precision_test = precision_score(y_test_cost, y_pred_test)
    recall_test = recall_score(y_test_cost, y_pred_test)
    f1_test = f1_score(y_test_cost, y_pred_test)

    tn, fp, fn, tp = confusion_matrix(
        y_test_cost,
        y_pred_test
    ).ravel()

    test_cost = 10.0 * fn + fp  # Διόρθωση πολλαπλασιασμού κόστους

    sens = cost_sensitivity(
        y_conf_cost.values,
        p_conf_cost
    )

    # 9. Baselines
    model_results = compare_models(
        X,
        y,
        groups=groups_col,
        spw=spw
    )

    # Εμφάνιση αποτελεσμάτων baseline
    display(model_results.round(4))

    # 10. Save validated Agent 1 (Απευθείας αποθήκευση με joblib)
    import os
    import joblib

    PACKAGE_DIR = '/content/agent1_package'
    os.makedirs(PACKAGE_DIR, exist_ok=True)

    package = {
        'model': model_split,
        'raw_features': ['Type'] + RAW_SENSORS,
        'type_map': TYPE_MAP,
        'best_params': best,
        'scale_pos_weight': spw,
        'calibration_method': 'isotonic',
        'conformal_qhats': qhats_split,
        'conformal_alpha': float(split_conf_results['alpha']),
        'cost_threshold': float(t_star_split),
        'cost_ratio': {'C_FN': 10.0, 'C_FP': 1.0},
        'validation_info': {
            'protocol': 'train / conformal-calibration / independent-test',
            'empirical_coverage': float(split_conf_results['coverage']),
            'uncertainty_rate': float(split_conf_results['uncertain_rate'])
        }
    }

    PKL_PATH = os.path.join(PACKAGE_DIR, 'agent1_validated_package.pkl')
    joblib.dump(package, PKL_PATH)

    # Φόρτωση μέσω της κλάσης για επαλήθευση
    validated_agent1 = Agent1FailureDetector(PKL_PATH)

    # Ασφαλής μορφοποίηση του qhat για το print (Mondrian dict support)
    qhat_print_str = (
        str({k: round(v, 4) for k, v in qhats_split.items()})
        if isinstance(qhats_split, dict)
        else f"{qhats_split:.6f}"
    )

    print("\n=== FINAL AGENT 1 RESULTS ===")
    print(f"Best params: {best}")
    print(f"Threshold: {t_star_split:.2f}")
    print(f"Precision: {precision_test:.4f}")
    print(f"Recall: {recall_test:.4f}")
    print(f"F1: {f1_test:.4f}")
    print(f"Test cost: {test_cost}")
    print(f"Conformal qhat: {qhat_print_str}")
    print(f"Coverage: {split_conf_results['coverage']:.4f}")
    print(f"Uncertainty: {split_conf_results['uncertain_rate']:.4f}")
    print("\nAgent 1 completed successfully.")


In [ ]:
# ---------------------------------------------------------------
# 18. Write Standalone Inference Script (predict.py) for Production Deployment
# --------------------------------------------------------------
%%writefile /content/agent1_package/predict.py

import os
import sys
import joblib
import numpy as np
import pandas as pd

from sklearn import set_config
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import KNNImputer
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier


# IMPORTANT:
# Same sklearn output configuration used during training
set_config(transform_output="pandas")


# ============================================================================
# Custom Transformers / Feature Engineering
# ============================================================================

class MissingIndicatorAdder:
    def __init__(self):
        pass
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X


class FeatureEngineer:
    def __init__(self):
        pass
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return engineer_features(X)


def engineer_features(X):
    X = X.copy()

    # Υπολογισμός διαφοράς θερμοκρασίας σε Kelvin (μόνο αυτό υπήρχε στο training)
    X['Temp_diff_K'] = (
        X['Process_temperature_K']
        - X['Air_temperature_K']
    )

    # Προσθήκη των missing indicators
    X['Air_temperature_K_isnan'] = X['Air_temperature_K'].isna().astype(int)
    X['Process_temperature_K_isnan'] = X['Process_temperature_K'].isna().astype(int)

    return X


# Ορισμός στο __main__ ώστε να επιτρέπεται το σωστό deserialization
import __main__
__main__.MissingIndicatorAdder = MissingIndicatorAdder
__main__.FeatureEngineer = FeatureEngineer
__main__.engineer_features = engineer_features


# ============================================================================
# Paths
# ============================================================================

if '__file__' in globals():
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
else:
    SCRIPT_DIR = os.getcwd()

MODEL_PATH = os.path.join(SCRIPT_DIR, 'agent1_validated_package.pkl')

if not os.path.exists(MODEL_PATH):
    colab_path = '/content/agent1_package/agent1_validated_package.pkl'
    if os.path.exists(colab_path):
        MODEL_PATH = colab_path


# ============================================================================
# Agent 1
# ============================================================================

class Agent1FailureDetector:
    """
    Agent 1 — Standalone Binary Failure Detection Inference Engine.
    """
    def __init__(self, package_path=MODEL_PATH):
        package = joblib.load(package_path)

        self.model = package['model']
        self.type_map = package['type_map']
        self.threshold = package['cost_threshold']

        self.qhat = package.get('conformal_qhats', package.get('conformal_qhat'))
        self.alpha = package.get('conformal_alpha', 0.10)
        self.calibration_method = package.get('calibration_method', 'isotonic')

    def predict(self, machine_data):
        data = machine_data.copy()

        # 1. Κωδικοποίηση τύπου μηχανής
        machine_type = data['Type']
        if isinstance(machine_type, str):
            machine_type = machine_type.upper()
            if machine_type not in self.type_map:
                raise ValueError(f"Unknown Type: {machine_type}. Expected {list(self.type_map.keys())}")
            machine_type = self.type_map[machine_type]
        else:
            machine_type = int(machine_type)

        # 2. Δημιουργία τυποποιημένου DataFrame εισόδου
        air_temp = data.get('Air_temperature_K', np.nan)
        proc_temp = data.get('Process_temperature_K', np.nan)

        X_new = pd.DataFrame([{
            'Type': machine_type,
            'Air_temperature_K': air_temp,
            'Process_temperature_K': proc_temp,
            'Rotational_speed_rpm': data.get('Rotational_speed_rpm', np.nan),
            'Torque_Nm': data.get('Torque_Nm', np.nan),
            'Tool_wear_min': data.get('Tool_wear_min', np.nan),
            'Air_temperature_K_isnan': int(pd.isna(air_temp)),
            'Process_temperature_K_isnan': int(pd.isna(proc_temp))
        }])

        # 3. Υπολογισμός πιθανότητας
        failure_probability = float(self.model.predict_proba(X_new)[0, 1])

        # 4. Cost-sensitive δυαδική απόφαση
        failure_detected = bool(failure_probability >= self.threshold)

        # 5. Mondrian Conformal Prediction set
        class_probs = np.array([1.0 - failure_probability, failure_probability])

        if isinstance(self.qhat, dict):
            current_qhat = self.qhat.get(machine_type, 0.1)
        else:
            current_qhat = self.qhat

        included = (1.0 - class_probs) <= current_qhat
        prediction_set = []
        if included[0]:
            prediction_set.append('NO FAILURE')
        if included[1]:
            prediction_set.append('FAILURE')

        uncertain = (len(prediction_set) != 1)
        if len(prediction_set) == 1:
            uncertainty_state = 'SINGLETON'
        elif len(prediction_set) == 0:
            uncertainty_state = 'EMPTY_SET'
        else:
            uncertainty_state = 'AMBIGUOUS_SET'

        # 6. Payload για τον Agent 2
        agent2_payload = None
        if failure_detected:
            agent2_payload = {
                'Type': machine_type,
                'Air_temperature_K': air_temp,
                'Process_temperature_K': proc_temp,
                'Rotational_speed_rpm': data.get('Rotational_speed_rpm', np.nan),
                'Torque_Nm': data.get('Torque_Nm', np.nan),
                'Tool_wear_min': data.get('Tool_wear_min', np.nan),
                'agent1_failure_probability': round(failure_probability, 6),
                'agent1_uncertain': uncertain,
                'agent1_uncertainty_state': uncertainty_state,
                'agent1_prediction_set': prediction_set
            }

        return {
            'prediction': 'FAILURE' if failure_detected else 'NO FAILURE',
            'failure_probability': round(failure_probability, 6),
            'failure_detected': failure_detected,
            'threshold': round(self.threshold, 6),
            'prediction_set': prediction_set,
            'uncertain': uncertain,
            'uncertainty_state': uncertainty_state,
            'conformal_alpha': self.alpha,
            'conformal_qhat': self.qhat,
            'calibration_method': self.calibration_method,
            'send_to_agent2': failure_detected,
            'agent2_payload': agent2_payload
        }


# ============================================================================
# Public interface
# ============================================================================

_agent = None


def predict_failure(machine_data):
    global _agent

    if _agent is None:
        _agent = Agent1FailureDetector()

    return _agent.predict(machine_data)

In [ ]:
# -------------------------------------------------------------------
# 18.1 Import and Test Prediction Module
# ------------------------------------------------------------------

%%writefile /content/agent1_package/test_agent1.py

import sys
import os

# Ασφαλής εύρεση διαδρομής (συμβατή και με Colab και με Terminal)
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = '/content/agent1_package'

if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from predict import predict_failure

def run_smoke_test():
    print("=" * 50)
    print("Εκτέλεση Δοκιμής Agent 1 (Smoke Test)...")
    print("=" * 50)

    sample_data = {
        'Type': 'M',
        'Air_temperature_K': 300.5,
        'Process_temperature_K': 311.2,
        'Rotational_speed_rpm': 1350,
        'Torque_Nm': 65.0,
        'Tool_wear_min': 200
    }

    try:
        result = predict_failure(sample_data)
        print("\n[ΕΠΙΤΥΧΙΑ] Η πρόβλεψη εκτελέστηκε κανονικά!")
        print("-" * 40)
        for key, value in result.items():
            print(f"  {key}: {value}")
        print("-" * 40)
    except Exception as e:
        print(f"\n[ΣΦΑΛΜΑ] Απέτυχε η πρόβλεψη: {e}")
        sys.exit(1)

if __name__ == '__main__':
    run_smoke_test()

In [ ]:
# -----------------------------------------------------------------
# 18.2 Write Package Dependencies (requirements.txt) for Deployment
# -----------------------------------------------------------------

%%writefile /content/agent1_package/requirements.txt
numpy
pandas
scikit-learn
imbalanced-learn
xgboost
cloudpickle

In [ ]:
# -------------------------------------------------------------------
# 18.3 Compress Deployment Package & Trigger Download (.zip)
# ------------------------------------------------------------------

import shutil
from google.colab import files

# Συμπίεση του φακέλου παραγωγής σε αρχείο zip
shutil.make_archive(
    '/content/agent1_package',
    'zip',
    '/content/agent1_package'
)
print("Deployment package successfully archived to '/content/agent1_package.zip'")

# Αυτόματο κατέβασμα αρχείου
files.download(
    '/content/agent1_package.zip'
)